In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import StatePreparation, XGate, YGate, ZGate, HGate, IGate

alphas = [1, 1, 1, 1]
lambda_val = sum(alphas)

amplitudes = [np.sqrt(a/lambda_val) for a in alphas]

prep = StatePreparation(amplitudes, label='PREP')
prep_inverse = prep.inverse()
prep_inverse.label = 'PREP_inverse'


anc = QuantumRegister(2, name='anc')
tgt = QuantumRegister(1, name='tgt')
cr = ClassicalRegister(2, name='meas')
cr2 = ClassicalRegister(1, name='tgt_meas')
qc = QuantumCircuit(anc, tgt, cr, cr2)
qc.x(tgt)
qc.h(tgt)
qc.append(prep, anc)
qc.barrier()


ch_gate = IGate().control(2, ctrl_state='00')
qc.append(ch_gate, [anc[0], anc[1], tgt[0]])

cx_gate = XGate().control(2, ctrl_state='01')
qc.append(cx_gate, [anc[0], anc[1], tgt[0]])

cy_gate = YGate().control(2, ctrl_state='10')
qc.append(cy_gate, [anc[0], anc[1], tgt[0]])


cz_gate = ZGate().control(2, ctrl_state='11')
qc.append(cz_gate, [anc[0], anc[1], tgt[0]])
qc.barrier()


qc.append(prep_inverse, anc)
qc.barrier()


qc.measure(anc, cr)
qc.measure(tgt, cr2)

# Draw the circuit
print(qc.draw('text'))

from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit import transpile
from matplotlib import pyplot as plt

simulator = AerSimulator(shots=100000)
compiled = transpile(qc, simulator).decompose()
job = simulator.run(compiled)
result = job.result()

counts = result.get_counts(compiled)

plot_histogram(counts)

plt.show()